# 시계열 데이터의 결측치 탐색 및 시각화 

```python
import pandas as pd 
import numpy as np

train_building_org = pd.read_csv("building_mart87_power_org.csv")
train_building_org['date_time'] = pd.to_datetime(train_building_org['date_time'])

display(train_building_org.head())

display(f"{train_building_org['date_time'].min().date()} ~ {train_building_org['date_time'].max().date()}")
```

---

### Result

```
	date_time	temp	wind	hum	power
0	2022-06-01 00:00:00	13.6	0.7	52.0	412.02
1	2022-06-01 01:00:00	13.4	0.0	57.0	413.82
2	2022-06-01 02:00:00	13.3	0.2	57.0	405.36
3	2022-06-01 03:00:00	11.9	0.6	66.0	392.58
4	2022-06-01 04:00:00	14.4	0.7	51.0	416.70
'2022-06-01 ~ 2022-08-24'
```

## 결측치 처리(선형 보간 법)

- `interpolate()` : 결측치를 지정된 방법(method)에 따라 채움. method = `linear` (선형 보간법)을 사용 
- `bfill()`

```python
df = train_building_org.copy()

# 결측치 처리: 선형 보간법 사용
df['wind_Interpolated'] = df['wind'].interpolate(method='linear')
# 선형 보간법 : 결측지가 있는 지점의 앞뒤 값을 직선으로 연결하여 중간 값을 채워 넣는 방식
# 데이터가 일정하게 변할 때 효과적 
# 결측치 처리: 전후 값으로 채우기
df['wind_Fill'] = df['wind'].bfill()
#bfill() : 결측치를 바로 다음에 있는유효한 값으로 채운다. 
#데이터가 일관되게 이어지는 경우 단기적 결측이 발생할 때 유용

# 결과 출력
print(df[['wind', 'wind_Interpolated', 'wind_Fill']].head())
```

```
                     wind  wind_Interpolated  wind_Fill
date_time                                              
2022-06-01 00:00:00   0.7                0.7        0.7
2022-06-01 01:00:00   0.0                0.0        0.0
2022-06-01 02:00:00   0.2                0.2        0.2
2022-06-01 03:00:00   0.6                0.6        0.6
2022-06-01 04:00:00   0.7                0.7        0.7
```




## 시계열 데이터의 다운 샘플링

- 시계열 데이터를 분석할 때, 더 큰 시간 간격으로 데이터를 요약하는 다운 샘플링 
- 다운샘플링을 통해 데이터의 노이즈를 줄이고, 전반적인 트랜드를 파악 할 수 있다. 

```python
weekly_data = df.resample('W').mean()
daily_data = df.resample('D').mean()

# 결과 출력
print("Daily Data:\n", daily_data.head())
print("Weekly Data:\n", weekly_data.head())

print(f"Daily Data: {weekly_data.index.min()} ~ {weekly_data.index.max()} ({len(daily_data)})")
print(f"Weekly Data: {weekly_data.index.min()} ~ {weekly_data.index.max()} ({len(weekly_data)})")
```

```
Daily Data:
                  temp      wind        hum      power  wind_Interpolated  \
date_time                                                                  
2022-06-01  20.279167  1.695833  44.958333  1093.0500           1.695833   
2022-06-02  19.325000  1.412500  69.333333  1077.9375           1.412500   
2022-06-03  22.816667  1.133333  67.791667  1121.5350           1.133333   
2022-06-04  22.837500  0.812500  64.750000  1166.5725           0.812500   
2022-06-05  21.404167  0.908333  69.000000  1059.2475           0.908333   

            wind_Fill  
date_time              
2022-06-01   1.695833  
2022-06-02   1.412500  
2022-06-03   1.133333  
2022-06-04   0.812500  
2022-06-05   0.908333  
Weekly Data:
                  temp      wind        hum        power  wind_Interpolated  \
date_time                                                                    
2022-06-05  21.332500  1.192500  63.166667  1103.668500           1.192500   
2022-06-12  19.267262  1.035119  75.714286   999.043929           1.035119   
2022-06-19  19.833929  0.934524  84.666667  1097.423571           0.934524   
2022-06-26  24.004167  1.459524  85.279762  1110.493929           1.459524   
2022-07-03  25.369811  2.151786  90.937500  1269.092143           2.151786   

            wind_Fill  
date_time              
2022-06-05   1.192500  
2022-06-12   1.035119  
2022-06-19   0.934524  
2022-06-26   1.459524  
2022-07-03   2.151786  
Daily Data: 2022-06-05 00:00:00 ~ 2022-08-28 00:00:00 (85)
Weekly Data: 2022-06-05 00:00:00 ~ 2022-08-28 00:00:00 (13)
```

# Upsampling

```python
minutely_data = df.resample('10min').interpolate()
print(len(df))
print(len(minutely_data))
print(minutely_data.head())
print(minutely_data.index)
```

```
2040
12235
                          temp      wind        hum   power  \
date_time                                                     
2022-06-01 00:00:00  13.600000  0.700000  52.000000  412.02   
2022-06-01 00:10:00  13.566667  0.583333  52.833333  412.32   
2022-06-01 00:20:00  13.533333  0.466667  53.666667  412.62   
2022-06-01 00:30:00  13.500000  0.350000  54.500000  412.92   
2022-06-01 00:40:00  13.466667  0.233333  55.333333  413.22   

                     wind_Interpolated  wind_Fill  
date_time                                          
2022-06-01 00:00:00           0.700000   0.700000  
2022-06-01 00:10:00           0.583333   0.583333  
2022-06-01 00:20:00           0.466667   0.466667  
2022-06-01 00:30:00           0.350000   0.350000  
2022-06-01 00:40:00           0.233333   0.233333  
DatetimeIndex(['2022-06-01 00:00:00', '2022-06-01 00:10:00',
               '2022-06-01 00:20:00', '2022-06-01 00:30:00',
               '2022-06-01 00:40:00', '2022-06-01 00:50:00',
               '2022-06-01 01:00:00', '2022-06-01 01:10:00',
               '2022-06-01 01:20:00', '2022-06-01 01:30:00',
               ...
               '2022-08-24 21:30:00', '2022-08-24 21:40:00',
               '2022-08-24 21:50:00', '2022-08-24 22:00:00',
               '2022-08-24 22:10:00', '2022-08-24 22:20:00',
               '2022-08-24 22:30:00', '2022-08-24 22:40:00',
               '2022-08-24 22:50:00', '2022-08-24 23:00:00'],
              dtype='datetime64[ns]', name='date_time', length=12235, freq='10T')
```

## 시계열 데이터의 다운 샘플링 시각화 및 해석 

```python
target_col = 'wind'

# 두 번째 그래프: 특정 기간 확대
start_date = start_date = pd.to_datetime('2022-07-13')
end_date = start_date + pd.Timedelta(days=14)  # 조절 가능

# 시각화
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 첫 번째 그래프: 전체 기간
ax1.plot(df.index, df[target_col], label='Original', linestyle='--')
ax1.plot(daily_data.index, daily_data[target_col], label='Daily (Resampled)', color='black', marker='o')
ax1.plot(weekly_data.index, weekly_data[target_col], label='Weekly (Resampled)', color='red', marker='o')
ax1.set_title("Full Period: Original, Daily, and Weekly Resampling")
ax1.legend()

# 확대 영역 표시
ax1.axvline(x=start_date, color='orange', linestyle='--', label='Zoom Start')
ax1.axvline(x=end_date, color='orange', linestyle='--', label='Zoom End')
# 그래프에 수직선을 그려 특정 기간의 종료 지점을 강조 
# x=start_date: 수직선이 start_date 위치에 그려지도록 설정
# linestyle: '-' 수직선을 점선 스타일로 그립니다. 
# label: Zoom Start 범례에 'Zoom Start'로 표시되도록 합니다. 이 라벨은 확대 분석할 기간의 시작을 나타냄 
ax1.legend()


ax2.plot(df.loc[start_date:end_date].index, df.loc[start_date:end_date, target_col], label='Original', linestyle='--')
ax2.plot(daily_data.loc[start_date:end_date].index, daily_data.loc[start_date:end_date, target_col], label='Daily (Resampled)', color='black', marker='o')
ax2.plot(weekly_data.loc[start_date:end_date].index, weekly_data.loc[start_date:end_date, target_col], label='Weekly (Resampled)', color='red', marker='o')
ax2.set_title(f"Zoomed Period ({start_date.date()} to {end_date.date()}): Original, Daily, and Weekly Resampling")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"전체 데이터 기간: {df.index.min().date()} to {df.index.max().date()}")
print(f"확대된 기간: {start_date.date()} to {end_date.date()}")
```

```
전체 데이터 기간: 2022-06-01 to 2022-08-24
확대된 기간: 2022-07-13 to 2022-07-27
```

## 시계열 데이터의 업샘플링 및 시각화 

```python
minutely_data = df.resample('10min').interpolate() #10분 간격으로 리 샘플링, 원본데이터가 시간 단위로 되어 있기 때문에, 더 짧은 10분 간격으로 나누어 데이터 생성 
# .interpolate : 리샘플링 된 시간 간격에 따라 생성된 새로운 데이터 포인트를 보간(interpolate)하여 값을 채움. 보간법은 원본 데이터의 앞뒤 값을 사용하여 새로운 데이터 포인트의 값을 추정( 선형 보간법 ) 
print(len(df))
print(len(minutely_data))
print(minutely_data.head())
print(minutely_data.index)
```

```
2040
12235
                          temp      wind        hum   power  \
date_time                                                     
2022-06-01 00:00:00  13.600000  0.700000  52.000000  412.02   
2022-06-01 00:10:00  13.566667  0.583333  52.833333  412.32   
2022-06-01 00:20:00  13.533333  0.466667  53.666667  412.62   
2022-06-01 00:30:00  13.500000  0.350000  54.500000  412.92   
2022-06-01 00:40:00  13.466667  0.233333  55.333333  413.22   

                     wind_Interpolated  wind_Fill  
date_time                                          
2022-06-01 00:00:00           0.700000   0.700000  
2022-06-01 00:10:00           0.583333   0.583333  
2022-06-01 00:20:00           0.466667   0.466667  
2022-06-01 00:30:00           0.350000   0.350000  
2022-06-01 00:40:00           0.233333   0.233333  
DatetimeIndex(['2022-06-01 00:00:00', '2022-06-01 00:10:00',
               '2022-06-01 00:20:00', '2022-06-01 00:30:00',
               '2022-06-01 00:40:00', '2022-06-01 00:50:00',
               '2022-06-01 01:00:00', '2022-06-01 01:10:00',
               '2022-06-01 01:20:00', '2022-06-01 01:30:00',
               ...
               '2022-08-24 21:30:00', '2022-08-24 21:40:00',
               '2022-08-24 21:50:00', '2022-08-24 22:00:00',
               '2022-08-24 22:10:00', '2022-08-24 22:20:00',
               '2022-08-24 22:30:00', '2022-08-24 22:40:00',
               '2022-08-24 22:50:00', '2022-08-24 23:00:00'],
              dtype='datetime64[ns]', name='date_time', length=12235, freq='10T')
```

```python
import matplotlib.pyplot as plt

# 시각화를 위한 날짜 범위 설정
start_date = pd.to_datetime('2022-07-13 00:00:00')
end_date = start_date + pd.Timedelta(hours=12)  # 2일간의 데이터를 보여줍니다

target_col = 'wind' #

# 시각화
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 첫 번째 그래프: 전체 기간
ax1.plot(df.loc[start_date:end_date].index, df.loc[start_date:end_date, target_col], 
         label='Original (Hourly)', marker='o', linestyle='--', color='blue', markersize=8)


ax2.plot(minutely_data.loc[start_date:end_date].index, minutely_data.loc[start_date:end_date, target_col], 
         label='5 Minute (Resampled)', marker='o',markersize=3, linestyle='--', alpha=0.7, color='purple', linewidth=1) 

# x축 레이블 회전
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"시각화 기간: {start_date.date()} to {end_date.date()}")
print(f"원본 데이터 포인트 수 (선택 기간): {len(df.loc[start_date:end_date])}")
print(f"업샘플링된 데이터 포인트 수 (선택 기간): {len(minutely_data.loc[start_date:end_date])}")
```

## 이동 평균을 사용한 시계열 데이터의 평활화 

- `simple moving average, SMA` 이동평균 
    - 시계열 데이터는 종종 변동성이 크고, 그로 인해 데이터의 전반적인 추세를 파악하기 어려울 수 있다. 
    - 변동성을 줄이고, 더 명확한 트랜드를 파악하기 위해 이동평균 사용 
- 이동평균은 특정 기간 내 데이터의 평균을 계산, 시계열 데이터의 급격한 변동을 부드럽게 만듬. 

```python
target_col = 'power'

num_window = 20
sma_col = f"SMA{num_window}"

# 이동 평균 계산 
daily_data[sma_col] = daily_data[target_col].rolling(window=num_window).mean()
# num_window=20, 이동 평균을 계산할 기간의 길이를 설정합니다. 여기서는 20일을 사용. 20일 동안의 데이터 평균을 계산 
# 결과 출력
print(daily_data[[target_col, sma_col]].head(num_window+5))

# 시각화
plt.figure(figsize=(12, 6))
plt.plot(daily_data.index,  daily_data[target_col], label='Original', linestyle='--')
plt.plot(daily_data.index,  daily_data[sma_col], label=sma_col)
plt.title(f"Sales Data: Original vs {sma_col}")
plt.legend()
plt.show()
```

- 이동평균 데이터는 원본데이터에 비해 훨씬 부드럽고, 전반적인 상승 또는 하락 트랜드를 더 명확하게 파악할 수 있다. 
- 단기적인 변동은 대부분 제거, 장기적인 추세를 이해하는데 유용 

## 지수 이동 평균 (EMA) 

- Exponential Moving Average, EMA
- 최근 데이터에 더 많은 가중치를 부여, 데이터의 최신 변동성을 더 민감하게 반영 

```python
selected_col = 'power'

num_window = 20

ema_col = f"EMA{num_window}"

# EMA 계산 
daily_data[ema_col] = daily_data[target_col].ewm(span=num_window, adjust=False).mean()

# 결과 출력
print(daily_data[[selected_col, sma_col, ema_col]].head(num_window+5))

# 시각화
plt.figure(figsize=(14, 7))
plt.plot(daily_data.index, daily_data[selected_col], label='Original', linestyle='--', color='blue', alpha=0.7)
plt.plot(daily_data.index, daily_data[sma_col], label=sma_col, color='orange', linewidth=2)
plt.plot(daily_data.index, daily_data[ema_col], label=ema_col, color='green', linewidth=2)
plt.title(f"Sales Data: Original vs {sma_col} vs {ema_col}")
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, alpha=0.3)

# x축 레이블 회전
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()
```

```
                power        SMA20        EMA20
date_time                                      
2022-06-01  1093.0500          NaN  1093.050000
2022-06-02  1077.9375          NaN  1091.610714
2022-06-03  1121.5350          NaN  1094.460646
2022-06-04  1166.5725          NaN  1101.328442
2022-06-05  1059.2475          NaN  1097.320733
2022-06-06  1030.1025          NaN  1090.918997
2022-06-07  1051.4175          NaN  1087.156949
2022-06-08  1105.2600          NaN  1088.881049
2022-06-09  1087.3350          NaN  1088.733807
2022-06-10  1115.6325          NaN  1091.295587
2022-06-11  1128.3075          NaN  1094.820531
2022-06-12   475.2525          NaN  1035.814052
2022-06-13  1123.4850          NaN  1044.163666
2022-06-14  1079.5200          NaN  1047.530936
2022-06-15   946.7025          NaN  1037.928228
2022-06-16  1077.5850          NaN  1041.705063
2022-06-17  1138.4025          NaN  1050.914343
2022-06-18  1182.7875          NaN  1063.473691
2022-06-19  1133.4825          NaN  1070.141197
2022-06-20  1190.6250  1069.212000  1081.615845
2022-06-21  1215.8775  1075.353375  1094.402669
2022-06-22  1216.3275  1082.272875  1106.014558
2022-06-23  1201.6500  1086.278625  1115.122695
2022-06-24  1191.8700  1087.543500  1122.431962
2022-06-25  1242.2775  1096.695000  1133.845823
```

## 주단위 최소값 날짜와 주말 여부 확인

아래 코드는 전체 시계열에서 주(week) 단위로 전력(`power`)이 가장 낮았던 시점과 그 값을 찾고,
그 날짜가 주말인지 여부까지 함께 분석합니다.

### 함수별 설명

- `copy()`
  - 원본 데이터프레임을 보존하기 위해 복사본을 생성합니다.
- `resample('W')`
  - DatetimeIndex를 기준으로 데이터를 주 단위(주말 기준)로 그룹화합니다.
- `agg(['idxmin', 'min'])`
  - `idxmin`: 각 주에서 최소값이 발생한 "원본 시각(index)"을 반환합니다.
  - `min`: 각 주의 최소 전력값 자체를 반환합니다.
- `apply(lambda x: x.dayofweek >= 5)`
  - `dayofweek`는 월요일=0, ..., 일요일=6입니다.
  - 따라서 5(토), 6(일)이면 `True`로 표시하여 주말 여부를 판별합니다.

```python
import pandas as pd

# 1) 원본 데이터 보존을 위해 복사
#    원본 train_building_org를 직접 건드리지 않고, 분석용 복사본 df를 사용합니다.
df = train_building_org.copy()

# 2) 날짜 컬럼을 인덱스로 설정(시계열 리샘플링을 위해 필수)
#    이미 인덱스가 date_time이면 이 줄은 생략 가능합니다.
df = df.set_index('date_time')

# 3) 원본(시간 단위) power에서 주 단위로 최소값 시점(idxmin)과 최소값(min) 계산
#    - idxmin: 그 주에서 power가 가장 작았던 "날짜/시간"
#    - min: 그 주의 최소 power 값
weekly_min_dates = df['power'].resample('W').agg(['idxmin', 'min'])

# 4) 최소값 발생 시점이 주말인지 여부 판별
#    dayofweek: 월=0, 화=1, 수=2, 목=3, 금=4, 토=5, 일=6
#    따라서 5 이상이면(토/일) 주말로 True 처리합니다.
weekly_min_dates['Is_Weekend'] = weekly_min_dates['idxmin'].apply(lambda x: x.dayofweek >= 5)

# 5) 결과 확인
print(weekly_min_dates)
```

### 참고

기존 코드에서 `daily_data = df.resample('W').mean()` 이후 다시
`daily_data['power'].resample('W')`를 적용하면 이미 주 단위로 축약된 데이터에 재집계를 하게 됩니다.
"원본 데이터에서 주별 최저 시점"을 찾으려면 위 코드처럼 원본 시계열(`df['power']`)에 바로 `resample('W')`를 적용하는 방식이 더 정확합니다.